# C - Conclude and Compare (Schlussfolgern und Vergleichen)

## QUA³CK-Phase

Phase C vergleicht experimentelle Ergebnisse anhand definierter quantitativer
und qualitativer Kriterien. Ziel ist nicht nur das kleinste Fehlermaß, sondern
eine begründete Entscheidung über Modellnutzen, Interpretierbarkeit,
Wartbarkeit und Grenzen.

## Umsetzung im Projekt

Das Gradient-Boosting-Modell wird gegen eine Median-Baseline auf einem
zeitlich nachgelagerten Testfenster verglichen. MAE, RMSE und R² bilden den
quantitativen Vergleich; Residuen, fachliche Plausibilität und
Implementierungsaufwand ergänzen die Entscheidung.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 30)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from pv_weather import TARGET, add_features, load_project_data, train_yield_model
from pv_weather.modeling import select_pv_relevant_hours

sns.set_theme(style="whitegrid")
data, source = load_project_data(
    ROOT / "data" / "processed" / "hourly_pv_weather.csv"
)
bundle = train_yield_model(data)
print(source)
print(f"Zeitlicher Test ab: {bundle.split_timestamp}")


## Quantitativer Modellvergleich


In [ ]:
comparison = pd.DataFrame(
    {
        "MAE (Prozentpunkte)": [
            bundle.metrics["baseline_mae"] * 100,
            bundle.metrics["model_mae"] * 100,
        ],
        "RMSE (Prozentpunkte)": [
            bundle.metrics["baseline_rmse"] * 100,
            bundle.metrics["model_rmse"] * 100,
        ],
        "R²": [
            bundle.metrics["baseline_r2"],
            bundle.metrics["model_r2"],
        ],
    },
    index=["Median-Baseline", "HistGradientBoosting"],
)
display(comparison.round(3))

mae_improvement = (
    1 - bundle.metrics["model_mae"] / bundle.metrics["baseline_mae"]
)
print(f"Relative MAE-Verbesserung gegenüber der Baseline: {mae_improvement:.1%}")


In [ ]:
residuals = pd.Series(bundle.residuals, name="Residuum")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(residuals * 100, bins=40, kde=True, ax=axes[0], color="#E19A18")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set(
    title="Verteilung der Testresiduen",
    xlabel="Beobachtung minus Prognose (Prozentpunkte)",
)
axes[1].scatter(
    np.arange(len(residuals)),
    residuals * 100,
    s=9,
    alpha=0.35,
    color="#376B5B",
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    title="Residuen in zeitlicher Reihenfolge",
    xlabel="Position im Testzeitraum",
    ylabel="Residuum (Prozentpunkte)",
)
plt.tight_layout()
plt.show()


## Qualitativer Vergleich


In [ ]:
qualitative_comparison = pd.DataFrame(
    [
        ("Vorhersagequalität", "niedrig", "höher; nichtlineare Zusammenhänge"),
        ("Interpretierbarkeit", "sehr hoch", "mittel; Permutationswichtigkeit verfügbar"),
        ("Trainingsaufwand", "sehr gering", "moderat"),
        ("Szenario-Plausibilität", "keine Wetterreaktion", "Monotoniebedingungen"),
        ("Wartbarkeit", "sehr einfach", "zentral in pv_weather/modeling.py gekapselt"),
    ],
    columns=["Kriterium", "Median-Baseline", "HistGradientBoosting"],
)
display(qualitative_comparison)


## Schlussfolgerung zur Forschungsfrage


In [ ]:
featured = add_features(data)
daylight = select_pv_relevant_hours(featured).dropna(subset=[TARGET])
high_radiation_limit = daylight["global_radiation_j_cm2"].quantile(0.75)
strong_sun = daylight[
    daylight["global_radiation_j_cm2"] >= high_radiation_limit
].copy()
strong_sun["Temperaturklasse"] = pd.cut(
    strong_sun["temperature_c"],
    [-np.inf, 15, 25, 30, np.inf],
    labels=["< 15 °C", "15-25 °C", "25-30 °C", ">= 30 °C"],
)
thesis_evidence = (
    strong_sun.groupby("Temperaturklasse", observed=True)[TARGET]
    .agg(Stunden="count", Mittelwert="mean", Median="median")
)
thesis_evidence[["Mittelwert", "Median"]] *= 100
display(thesis_evidence.round(2))

best_class = thesis_evidence["Median"].idxmax()
print(
    f"Bei starker Einstrahlung (>= {high_radiation_limit:.1f} J/cm²) "
    f"hat die Klasse {best_class} den höchsten beobachteten Median."
)


Die Tabelle liefert einen deskriptiven Hinweis zur These. Ein niedrigerer
Median bei sehr hohen Temperaturen wäre mit thermischen Verlusten vereinbar,
beweist sie aber nicht. Gleichzeitig verändern sich weitere Wetter- und
Betriebsbedingungen. Die modellierte Temperatur-Sensitivität ist ebenfalls
eine kontrollierte Modellreaktion und kein experimenteller Kausalnachweis.

## Modellentscheidung

Das Gradient-Boosting-Modell wird für die Anwendung gewählt, **wenn sein MAE im
zeitlichen Test unter dem MAE der Median-Baseline liegt**. Die Baseline bleibt
als dauerhafter Kontrollpunkt erhalten.


In [ ]:
model_selected = bundle.metrics["model_mae"] < bundle.metrics["baseline_mae"]
decision = "HistGradientBoosting auswählen" if model_selected else "Modell nicht freigeben"
print(decision)
assert model_selected, "Das Modell übertrifft die Baseline im aktuellen Test nicht."


## Grenzen und Risiken

- Das DWD-Stationsmittel ist nicht vollständig nach regionaler PV-Leistung
  gewichtet.
- Die geschätzte Modultemperatur ist keine direkte Messung.
- Jahreskapazitäten ignorieren unterjährigen Ausbau.
- Seltene sehr heiße Stunden besitzen größere statistische Unsicherheit.
- Anlagenneigung, Ausrichtung, Schnee, Verschattung, Abregelung und technische
  Verfügbarkeit fehlen.
- Testresiduen liefern ein empirisches 80-%-Intervall, aber keine vollständige
  probabilistische Prognose.

## Technische Verankerung der C-Phase

- `pv_weather/modeling.py`: zeitlicher Split, Baseline, Metriken und Residuen
- `tests/test_modeling.py`: automatisierte Modellverträge
- `app.py`: Darstellung von Modellgüte, Szenarien und Grenzen
- `notebooks/pv_wetter_deutschland.ipynb`: zusammenhängende Gesamtdokumentation

## Übergabe an K

Die K-Phase bereitet Entscheidung, Kernergebnisse, Einschränkungen und
Nutzungsanleitung zielgruppengerecht auf und überführt die gemeinsame
Kernlogik in die Streamlit-Anwendung.
